# Job ETL da silver

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. No caso desse projeto, a fonte será extraida da camada raw para a silver pelo arquivo Complete_Pokedex_V1.1.csv e o resultado será armazenado em outro csv e utilizado na camada gold.

### Frameworks utilizados

In [1]:
import pandas as pd
import psycopg2
import time

### Extrair

In [2]:
df = pd.read_csv('../Data_Layer/raw/Complete_Pokedex_V1.1.csv')

### Transformar


In [3]:
# Apaga colunas
colunas_para_apagar = [
'ability_1',
'ability_2',
'ability_3',
'number_pokemon_with_typing',
'primary_color',
'mean', 
'standard_deviation', 
'exp_to_level_100', 
'can_evolve', 
'final_evolution', 
'is_default', 
'baby_pokemon', 
'genus', 
'egg_group_1', 
'egg_group_2', 
'shape', 
'bmi', 
'special_attack', 
'special_defense', 
'speed'
]

df_tratado = df.drop(columns=colunas_para_apagar)

# Remove tuplas duplicadas do Pokedex_number 
df_tratado = df_tratado.drop_duplicates(subset=['pokedex_number'])

# Trata nulos, transforma para vazio
df_tratado[["type_2", "evolves_from"]] = df_tratado[["type_2", "evolves_from"]].fillna(" ")


print(df_tratado.head())

print("\n transformação concluída!")

   pokedex_number   pokemon_name type_1  type_2  height  weight  hit_points  \
0               1      Bulbasaur  Grass  Poison     0.7     6.9          45   
1               2        Ivysaur  Grass  Poison     1.0    13.0          60   
2               3  Mega Venusaur  Grass  Poison     2.4   155.5          80   
5               4     Charmander   Fire             0.6     8.5          39   
6               5     Charmeleon   Fire             1.1    19.0          58   

   attack  defense  total_stats  ...  against_ground  against_flying  \
0      49       49          318  ...             1.0             2.0   
1      62       63          405  ...             1.0             2.0   
2     100      123          625  ...             1.0             2.0   
5      52       43          309  ...             2.0             1.0   
6      64       58          405  ...             2.0             1.0   

   against_psychic  against_bug against_rock against_ghost  against_dragon  \
0             

### Carregar 

##### salva dado tratado carregando em um novo csv

In [4]:
df_tratado = df_tratado.to_csv("Complete_Pokedex-Tratada.csv", index=False)

##### popula dados no banco

In [5]:
# Conecta no PostgreSQL
while True:
    try:
        conexao = psycopg2.connect(
            host="postgres", # ou conectar localhost ou pode ser mesmo o postgres
            port=5432,
            database="pokedex_db",
            user="pokedex_user",
            password="pokedex_password"
        )
        break
    except psycopg2.OperationalError:
        print("O banco não está pronto, aguardando 3 segundos...")
        time.sleep(3)

# cria o cursor
cursor = conexao.cursor()

# com o cursos cria a tabela 

cursor.execute("""
CREATE TABLE IF NOT EXISTS pokemon (
    pokedex_number INT NOT NULL PRIMARY KEY,
    pokemon_name VARCHAR(50) NOT NULL,
    type_1 VARCHAR(50) NOT NULL,
    type_2 VARCHAR(50),
    height DOUBLE PRECISION NOT NULL,
    weight DOUBLE PRECISION NOT NULL,
    hit_points INT NOT NULL,
    attack INT NOT NULL,
    defense INT NOT NULL,
    total_stats INT NOT NULL,
    capture_rate INT NOT NULL,
    generation INT NOT NULL,
    base_happiness INT NOT NULL,
    base_experience INT NOT NULL,
    exp_type VARCHAR(50) NOT NULL,
    evolves_from VARCHAR(50),
    mega_evolution BOOLEAN NOT NULL,
    alolan_form BOOLEAN NOT NULL,
    galarian_form BOOLEAN NOT NULL,
    forms_switchable BOOLEAN NOT NULL,
    legendary BOOLEAN NOT NULL,
    mythical BOOLEAN NOT NULL,
    genderless BOOLEAN NOT NULL,
    female_rate DOUBLE PRECISION NOT NULL,
    egg_cycles INT NOT NULL,
    against_normal DOUBLE PRECISION NOT NULL,
    against_fire DOUBLE PRECISION NOT NULL,
    against_water DOUBLE PRECISION NOT NULL,
    against_electric DOUBLE PRECISION NOT NULL,
    against_grass DOUBLE PRECISION NOT NULL,
    against_ice DOUBLE PRECISION NOT NULL,
    against_fighting DOUBLE PRECISION NOT NULL,
    against_poison DOUBLE PRECISION NOT NULL,
    against_ground DOUBLE PRECISION NOT NULL,
    against_flying DOUBLE PRECISION NOT NULL,
    against_psychic DOUBLE PRECISION NOT NULL,
    against_bug DOUBLE PRECISION NOT NULL,
    against_rock DOUBLE PRECISION NOT NULL,
    against_ghost DOUBLE PRECISION NOT NULL,
    against_dragon DOUBLE PRECISION NOT NULL,
    against_dark DOUBLE PRECISION NOT NULL,
    against_steel DOUBLE PRECISION NOT NULL,
    against_fairy DOUBLE PRECISION NOT NULL
)
""")

conexao.commit()

# Populando os dados do CSV para o banco
with open('Complete_Pokedex-Tratada.csv', 'r') as arqCSV:
    next(arqCSV)  
    cursor.copy_from(arqCSV, 'pokemon', sep=',')

conexao.commit()

# Fechando conexão
cursor.close()
conexao.close()

print("concluido!")

concluido!


# PARTE DO PROFESSOR

In [6]:
import pandas as pd

df = pd.read_csv('../Data_Layer/raw/Complete_Pokedex_V1.1.csv')


In [7]:
df.head()

,pokedex_number,pokemon_name,type_1,type_2,ability_1,ability_2,ability_3,number_pokemon_with_typing,primary_color,shape,...,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,1,Bulbasaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,3,Mega Venusaur,Grass,Poison,Thick Fat,NaN,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,3,Venusaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,3,Venusaur Gmax,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5


In [8]:
df.columns

Index(['pokedex_number', 'pokemon_name', 'type_1', 'type_2', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles', 'against_normal',
       'against_fire', 'against_water', 'against_electric', 'against_grass',
       'against_ice', 'against_fighting', 'against_poison', 'against_ground',
       'against_flying', 'against_psychic', 'against_bug', 'against_rock',
       'against_ghost', '

In [9]:
df['type_1'].unique()

array(['Grass', 'Fire', 'Water', 'Bug', 'Normal', 'Dark', 'Poison',
       'Electric', 'Ice', 'Ground', 'Fairy', 'Steel', 'Fighting',
       'Psychic', 'Rock', 'Ghost', 'Dragon', 'Flying'], dtype=object)

In [10]:
df.columns

Index(['pokedex_number', 'pokemon_name', 'type_1', 'type_2', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles', 'against_normal',
       'against_fire', 'against_water', 'against_electric', 'against_grass',
       'against_ice', 'against_fighting', 'against_poison', 'against_ground',
       'against_flying', 'against_psychic', 'against_bug', 'against_rock',
       'against_ghost', '

In [11]:
df_temp = df[df['type_1'] == 'Grass']

In [12]:
df_tipo = df.drop(columns = ['pokedex_number', 'pokemon_name', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles',])

In [13]:
df_tipo

,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,Ice,NaN,1.0,2.0,1.0,1.0,1.00,0.5,2.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0
1114,Ghost,NaN,0.0,1.0,1.0,1.0,1.00,1.0,0.0,0.5,1.0,1.0,1.0,0.5,1.0,2.0,1.0,2.0,1.0,1.0
1115,Psychic,Grass,1.0,2.0,0.5,0.5,0.50,2.0,0.5,2.0,0.5,2.0,0.5,4.0,1.0,2.0,1.0,2.0,1.0,1.0
1116,Psychic,Ice,1.0,2.0,1.0,1.0,1.00,0.5,1.0,1.0,1.0,1.0,0.5,2.0,2.0,2.0,1.0,2.0,2.0,1.0


In [14]:
df_tipo = df_tipo.fillna('')

In [15]:
df_tipo['Tipo'] = df_tipo['type_1'] + ' ' + df_tipo['type_2']

In [16]:
df_tipo

,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,...,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy,Tipo
0,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
1,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
2,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
3,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
4,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,Ice,,1.0,2.0,1.0,1.0,1.00,0.5,2.0,1.0,...,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,Ice
1114,Ghost,,0.0,1.0,1.0,1.0,1.00,1.0,0.0,0.5,...,1.0,1.0,0.5,1.0,2.0,1.0,2.0,1.0,1.0,Ghost
1115,Psychic,Grass,1.0,2.0,0.5,0.5,0.50,2.0,0.5,2.0,...,2.0,0.5,4.0,1.0,2.0,1.0,2.0,1.0,1.0,Psychic Grass
1116,Psychic,Ice,1.0,2.0,1.0,1.0,1.00,0.5,1.0,1.0,...,1.0,0.5,2.0,2.0,2.0,1.0,2.0,2.0,1.0,Psychic Ice


In [17]:
df_tipo = df_tipo.drop_duplicates()

In [18]:
df_tipo.columns

Index(['type_1', 'type_2', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying', 'against_psychic',
       'against_bug', 'against_rock', 'against_ghost', 'against_dragon',
       'against_dark', 'against_steel', 'against_fairy', 'Tipo'],
      dtype='object')

In [19]:
df_tipo = df_tipo[['Tipo','type_1', 'type_2', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying', 'against_psychic',
       'against_bug', 'against_rock', 'against_ghost', 'against_dragon',
       'against_dark', 'against_steel', 'against_fairy']]

In [20]:
df_tipo.reset_index(inplace = True,drop=True)

In [21]:
df_tipo.head()

,Tipo,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,...,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,Grass Poison,Grass,Poison,1.0,2.00,0.5,0.5,0.25,2.0,0.5,...,1.0,2.0,2.0,1.00,1.0,1.0,1.0,1.0,1.0,0.5
1,Fire,Fire,,1.0,0.50,2.0,1.0,0.50,0.5,1.0,...,2.0,1.0,1.0,0.50,2.0,1.0,1.0,1.0,0.5,0.5
2,Fire Flying,Fire,Flying,1.0,0.50,2.0,2.0,0.25,1.0,0.5,...,0.0,1.0,1.0,0.25,4.0,1.0,1.0,1.0,0.5,0.5
3,Fire Dragon,Fire,Dragon,1.0,0.25,1.0,0.5,0.25,1.0,1.0,...,2.0,1.0,1.0,0.50,2.0,1.0,2.0,1.0,0.5,1.0
4,Water,Water,,1.0,0.50,0.5,2.0,2.00,0.5,1.0,...,1.0,1.0,1.0,1.00,1.0,1.0,1.0,1.0,0.5,1.0
